**Author**: Felipe Matheus  
**Surrogate**: annealing — tensile strength (UTS)

Didactic, step-by-step notebook for ONE UTS run (mirrors `annealing_iacs_fixed.ipynb`).
Differences from the IACS notebook, all driven by the UTS data:
- **No essay/schema rows yet** → no `is_essay`, no literature/schema concat; weights are uniform.
- **No held-out validation set yet** (`df_val = None`) → Section 8.1 degrades gracefully.
- **No physical ceiling** like 106 %IACS → only positivity (`Y_MIN = 0`, `Y_MAX = inf`), so truncation is a no-op.
- **5 features** including `initial_diameter` and the recursive `tensile_strength`.

For sweeping many configs use `run_experiments_uts.ipynb`; this one exposes every step.

# 1. Setup

In [ ]:
import os
import sys
import pickle

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from scipy.stats import norm
from autogluon.tabular import TabularPredictor


module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.processing.Processing import Processing
from src.feature_engineering.FeatureEngineering import FeatureEngineering
from src.modeling.Modeling import Modeling
from src.modeling.Evaluation import Evaluation
from src.DataLoader import LoaderHelper

from config.Variables import Variables

%load_ext autoreload
%autoreload 2

In [ ]:
# Helper instances reused across the notebook.
proc = Processing()
feng = FeatureEngineering()
modl = Modeling()
evla = Evaluation()
varv = Variables()
load = LoaderHelper()

## 1.1 Global configuration

In [ ]:
# ---- Experiment settings ----
TAG = "full-features_no-weight"

# ---- Paths ----
FILE_NAME = "dataset_annealing_tensile-strength.csv"
PROCESS = "annealing_uts"

# All artifacts of THIS run (predictors included) live under the TAG folder,
# so different experiments never overwrite each other's models.
EXPERIMENT_DIR = os.path.join(varv.PATHS.models, PROCESS, TAG)
os.makedirs(EXPERIMENT_DIR, exist_ok=True)

# ---- Dataset schema ----
TARGET = "tensile_strength_final"
FEATURES = ["purity", "initial_diameter", "tensile_strength", "temperature", "time"]
# No essay rows for UTS yet -> uniform weights (kept at 1.0).
WEIGHT_ON_ESSAY_ROWS = 1

# ---- AutoGluon training knobs (Model A) ----
PRESETS_A = "medium_quality"     # 'best_quality' for final runs only
NUM_BAG_FOLDS_A = 5              # K-fold bagging produces OOF preds
NUM_BAG_SETS_A = 1
NUM_STACK_LEVELS_A = 0           # no multi-level stacking; keep simple
TIME_LIMIT_A = 120

# ---- AutoGluon training knobs (Model B) ----
PRESETS_B = "medium_quality"
NUM_BAG_FOLDS_B = 5
NUM_STACK_LEVELS_B = 0
TIME_LIMIT_B = 60

# ---- Uncertainty pipeline knobs ----
USE_WEIGHTED_VARIANCE = True     # False = uniform mean/var (sanity check)
VARIANCE_FLOOR_FRAC = 0.01       # floor of sigma2_aleat as fraction of Var(y)
CALIBRATION_ALPHAS = (0.5, 0.8, 0.9, 0.95)
RECALIBRATION_TARGET_ALPHA = 0.9

# ---- AGL x H2O comparison ----
# No essay rows -> no stratification column; shared folds still available
# (plain KFold, since add_stratified_fold_column falls back when is_essay
# is absent).
USE_SHARED_FOLDS = True
FOLD_SEED = 42

# ---- Physical constraints (inference-time) ----
# UTS has no fixed ceiling like 106 %IACS. The only hard physical bound is
# positivity, and observed values sit far from 0, so truncation is a no-op.
# Kept explicit for symmetry with the IACS pipeline.
Y_MIN = 0.0
Y_MAX = np.inf

# 2. Data

UTS has a single literature dataset (no schema/essay split yet), so there is no
concat step. Rows missing the target are dropped.

In [ ]:
# Raw CSV.
df_raw = pd.read_csv(os.path.join(varv.PATHS.data_raw, FILE_NAME))
df_raw

In [ ]:
# Cast to float, label material elements, mark rows where ratios are valid.
df_float = proc.df_to_float(
    df_raw, drop_cols=["DOI", "is_Cu"], ignore_columns=["material"]
)
df_labeled = feng.label_element(df_float).drop_duplicates()
df_with_masks = feng.add_ratio_mask_column(
    feng.add_ratio_mask_column(df_labeled, "grain_size"),
    "tensile_strength",
)

df = (
    df_with_masks[FEATURES + [TARGET]]
    .dropna(subset=[TARGET])
    .reset_index(drop=True)
)

assert df[FEATURES + [TARGET]].isna().sum().sum() == 0, "NaNs in inputs"
print(f"Dataset: {df.shape}")
df.head()

## 2.1 Weights & folds

In [ ]:
# No essay rows for UTS yet -> uniform weights. weight_col is kept so the
# fit_model_a sample_weight path stays identical to the IACS pipeline.
df["weight_col"] = 1.0

# Optional shared fold column: the SAME partition is passed to AutoGluon
# (groups=) and H2O (fold_column=). add_stratified_fold_column falls back to
# plain KFold automatically when there is no is_essay column to stratify on.
if USE_SHARED_FOLDS:
    df = feng.add_stratified_fold_column(
        df, n_folds=NUM_BAG_FOLDS_A, seed=FOLD_SEED,
    )

df.head()

## 2.2 Validation data

No held-out UTS essays exist yet. When they arrive, load them into `df_val`
(same columns as `df`) and Section 8.1 will report validation metrics
automatically. Until then `df_val = None` and Section 8.1 is skipped.

In [ ]:
df_val = None

# 3. Train Model A — Mean Predictor with Bagging

In [ ]:
predictor_a = modl.fit_model_a(
    df=df,
    target=TARGET,
    features=FEATURES,
    path=os.path.join(EXPERIMENT_DIR, "model_a"),
    presets=PRESETS_A,
    num_bag_folds=NUM_BAG_FOLDS_A,
    num_bag_sets=NUM_BAG_SETS_A,
    num_stack_levels=NUM_STACK_LEVELS_A,
    time_limit=TIME_LIMIT_A,
    groups="fold_id" if USE_SHARED_FOLDS else None,
    sample_weight="weight_col",
)

In [ ]:
# Inspect the leaderboard. Last row is WeightedEnsemble_L2; the rest are
# the base learners whose OOF predictions feed the epistemic variance.
predictor_a.leaderboard(silent=True)

## 3.1 H2O AutoML Comparison

In [ ]:
import h2o
from h2o.automl import H2OAutoML

h2o.init(nthreads=-1, max_mem_size="4G")

h2o_cols = FEATURES + [TARGET, "weight_col"]
if USE_SHARED_FOLDS:
    h2o_cols.append("fold_id")
train_h2o = h2o.H2OFrame(df[h2o_cols])
train_h2o[TARGET] = train_h2o[TARGET].asnumeric()
if USE_SHARED_FOLDS:
    # fold_column must be categorical on the H2O side.
    train_h2o["fold_id"] = train_h2o["fold_id"].asfactor()

aml_kwargs = dict(
    max_runtime_secs=TIME_LIMIT_A,
    keep_cross_validation_predictions=True,
    exclude_algos=["DeepLearning"],
    sort_metric="RMSE",
    seed=42,
)
if not USE_SHARED_FOLDS:
    aml_kwargs["nfolds"] = NUM_BAG_FOLDS_A  # ignored/conflicting with fold_column

aml_h2o = H2OAutoML(**aml_kwargs)

train_kwargs = dict(x=FEATURES, y=TARGET, training_frame=train_h2o,
                    weights_column="weight_col")
if USE_SHARED_FOLDS:
    train_kwargs["fold_column"] = "fold_id"

aml_h2o.train(**train_kwargs)

aml_h2o.leaderboard.head(10)

In [ ]:
# The coherent OOF-vs-OOF comparison (unweighted metrics, same folds) is in
# Section 10. AutoGluon's score_val vs H2O's weighted xval RMSE would be
# apples-to-oranges, so it is intentionally not used here.
print("See Section 10 for the coherent OOF-vs-OOF comparison.")

# 4. OOF Predictions and Ensemble Weights

In [ ]:
# OOF mean predictions of the final ensemble.
mu_oof = predictor_a.predict_oof()                  # pd.Series, index == df.index
residuals_oof = df[TARGET] - mu_oof
print(f"mu_oof: shape={mu_oof.shape}  any NaN? {mu_oof.isna().any()}")

## 4.1 Fast check on big residuals

In [ ]:
residuals_oof.describe()

In [ ]:
# Rows where the ensemble is off by a large UTS margin (units: MPa).
df[residuals_oof.abs() > 50]

## 4.2 Recovering weights

In [ ]:
# OOF matrix per base learner (excludes WeightedEnsemble itself).
oof_matrix, base_model_names = modl.collect_oof_base_learners(predictor_a)
print(f"oof_matrix shape: {oof_matrix.shape}  (M base learners, N rows)")
print("Base learners:", base_model_names)

In [ ]:
# Recover ensemble weights by solving (oof_matrix.T @ w = mu_oof) with w >= 0.
weights, is_recovery_ok, max_diff = modl.recover_ensemble_weights(
    oof_matrix=oof_matrix,
    mu_oof_ensemble=mu_oof.values,
)

print(f"Recovery max diff: {max_diff:.6f}  ({'OK' if is_recovery_ok else 'FALLBACK'})")
print("\nNon-zero weights:")
for name, w in zip(base_model_names, weights):
    if w > 1e-4:
        print(f"  {name:30s}  {w:.4f}")

# 5. Compute $\hat\mu$ and $\hat\sigma^2_{\text{epist}}$

In [ ]:
mu_recomputed, sigma2_epist_oof = modl.compute_mu_and_epistemic_variance(
    preds_matrix=oof_matrix,
    weights=weights,
    use_weights=USE_WEIGHTED_VARIANCE,
)

mu_diffs = np.abs(mu_recomputed - mu_oof)
mu_diffs.describe()

In [ ]:
plt.figure(figsize=(7, 3))
plt.hist(np.sqrt(sigma2_epist_oof), bins=30)
plt.xlabel("epistemic std (sqrt sigma2_epist), MPa")
plt.ylabel("count")
plt.title("Epistemic uncertainty across OOF rows")
plt.tight_layout()
plt.show()

# 6. Build Aleatoric Targets $\tilde r^2$

In [ ]:
r_tilde_sq, residuals_oof_arr, diag = modl.build_aleatoric_targets(
    y_true=df[TARGET].values,
    mu_oof=mu_oof.values,
    sigma2_epist_oof=sigma2_epist_oof,
)

print("=== Aleatoric target diagnostics ===")
for k, v in diag.items():
    print(f"  {k:20s} = {v}")

# 7. Train Model B — Aleatoric Variance Predictor

In [ ]:
predictor_b = modl.fit_model_b(
    df=df,
    features=FEATURES,
    r_tilde_sq=r_tilde_sq,
    path=os.path.join(EXPERIMENT_DIR, "model_b"),
    presets=PRESETS_B,
    num_bag_folds=NUM_BAG_FOLDS_B,
    num_stack_levels=NUM_STACK_LEVELS_B,
    time_limit=TIME_LIMIT_B,
)

In [ ]:
predictor_b.leaderboard(silent=True)

# 8. Calibration Diagnostics

In [ ]:
df[TARGET].var()

In [ ]:
variance_floor = VARIANCE_FLOOR_FRAC * df[TARGET].var()
sigma2_aleat_oof = modl.predict_aleatoric_variance(
    predictor_b, df[FEATURES], variance_floor=variance_floor,
)

sigma2_total_oof = sigma2_epist_oof + sigma2_aleat_oof
sigma_total_oof = np.sqrt(sigma2_total_oof)

y_true = df[TARGET].values
cal_before = modl.calibration_table(
    mu=mu_oof.values, sigma=sigma_total_oof, y_true=y_true,
    alphas=CALIBRATION_ALPHAS,
)
print("=== Calibration BEFORE recalibration ===")
cal_before

Fit a scalar recalibration factor to pull coverage using α=0.9 as nominal reference.

In [ ]:
c_opt = modl.fit_recalibration_scalar(
    mu=mu_oof.values,
    sigma=sigma_total_oof,
    y_true=y_true,
    target_alpha=RECALIBRATION_TARGET_ALPHA,
)
print(f"Recalibration scalar c = {c_opt:.4f}")

cal_after = modl.calibration_table(
    mu=mu_oof.values, sigma=c_opt * sigma_total_oof, y_true=y_true,
    alphas=CALIBRATION_ALPHAS,
)
print("\n=== Calibration AFTER recalibration ===")
cal_after

# 8.1 Validation Set Evaluation

Evaluate the freshly trained pair (A + B, with recalibration $c$) on a held-out
`df_val`. UTS has none yet, so this section is guarded by `if df_val is not None`.
The moment essays arrive, set `df_val` in Section 2.2 and these cells report
validation rmse/mae/coverage with no other change. `modl.predict_target` is
target-agnostic (DataFrame or dict, any AutoGluon predictor).

In [ ]:
if df_val is not None:
    preds_val = modl.predict_with_uncertainty(
        X=df_val,
        predictor_a=predictor_a,
        predictor_b=predictor_b,
        weights=weights,
        model_names=base_model_names,
        features=FEATURES,
        variance_floor=variance_floor,
        recalibration_c=c_opt,
        use_weights=USE_WEIGHTED_VARIANCE,
        y_min=Y_MIN,
        y_max=Y_MAX,
    )
    val_metrics = evla.metrics_on_dataframe(
        df=df_val,
        target=TARGET,
        mu=preds_val["mu"].to_numpy(),
        sigma=preds_val["sigma_total"].to_numpy(),
        alphas=CALIBRATION_ALPHAS,
    )
    print("=== Validation metrics ===")
    print({k: round(v, 4) for k, v in val_metrics.items() if k != "coverage"})
    print("coverage:", {a: round(c, 3) for a, c in val_metrics["coverage"].items()})
else:
    print("No df_val yet for UTS - skipping validation metrics.")
    val_metrics = None

In [ ]:
# ---- Target-agnostic point prediction from a plain dict ----
single = modl.predict_target(
    predictor_a,
    {"purity": 99.99, "initial_diameter": 23.0, "tensile_strength": 1200.0,
     "temperature": 473.0, "time": 60.0},
)
print(f"Predicted {TARGET}: {single[0]:.1f} MPa")

# 9. Inference Example

Full predictive output with every mechanism on: epistemic + aleatoric variance,
OOD inflation, recalibration. Truncation columns (`mu_trunc`, `sigma_trunc`,
`p_above_max`) only appear when `Y_MAX`/`Y_MIN` are finite; for UTS `Y_MAX=inf`,
so the truncation block is a no-op and those columns are omitted.

In [ ]:
df.head(5)

In [ ]:
# Fit the OOD reference on the training inputs (stored in artifacts later).
ood_ref = modl.fit_ood_reference(df, features=FEATURES)

preds = modl.predict_with_uncertainty(
    X=df.head(5),
    predictor_a=predictor_a,
    predictor_b=predictor_b,
    weights=weights,
    model_names=base_model_names,
    features=FEATURES,
    variance_floor=variance_floor,
    recalibration_c=c_opt,
    use_weights=USE_WEIGHTED_VARIANCE,
    y_min=Y_MIN,
    y_max=Y_MAX,          # inf -> no truncation columns for UTS
    ood_ref=ood_ref,      # inputs outside training cloud get inflated sigma
)
preds

# 10. Persist Artifacts

In [ ]:
# ---- Experiment artifacts (everything under EXPERIMENT_DIR / TAG) ----
artifacts_path = os.path.join(EXPERIMENT_DIR, "artifacts.pkl")

artifacts = {
    "features": FEATURES,
    "target": TARGET,
    "base_model_names": base_model_names,
    "weights": weights,
    "variance_floor": variance_floor,
    "recalibration_c": c_opt,
    "use_weighted_variance": USE_WEIGHTED_VARIANCE,
    "calibration_before": cal_before,
    "calibration_after": cal_after,
    "y_min": Y_MIN,
    "y_max": Y_MAX,
    "ood_ref": ood_ref,
}
with open(artifacts_path, "wb") as f:
    pickle.dump(artifacts, f)

print(f"Saved to: {artifacts_path}")

In [ ]:
# ---- Save global configuration (1.1) as YAML for this experiment (TAG) ----
global_config = {
    "TAG": TAG,
    "varv.PATHS.data_raw": varv.PATHS.data_raw,
    "varv.PATHS.models": varv.PATHS.models,
    "FILE_NAME": FILE_NAME,
    "PROCESS": PROCESS,
    "TARGET": TARGET,
    "FEATURES": FEATURES,
    "WEIGHT_ON_ESSAY_ROWS": WEIGHT_ON_ESSAY_ROWS,
    "PRESETS_A": PRESETS_A,
    "NUM_BAG_FOLDS_A": NUM_BAG_FOLDS_A,
    "NUM_BAG_SETS_A": NUM_BAG_SETS_A,
    "NUM_STACK_LEVELS_A": NUM_STACK_LEVELS_A,
    "TIME_LIMIT_A": TIME_LIMIT_A,
    "PRESETS_B": PRESETS_B,
    "NUM_BAG_FOLDS_B": NUM_BAG_FOLDS_B,
    "NUM_STACK_LEVELS_B": NUM_STACK_LEVELS_B,
    "TIME_LIMIT_B": TIME_LIMIT_B,
    "USE_WEIGHTED_VARIANCE": USE_WEIGHTED_VARIANCE,
    "VARIANCE_FLOOR_FRAC": VARIANCE_FLOOR_FRAC,
    "CALIBRATION_ALPHAS": list(CALIBRATION_ALPHAS),
    "RECALIBRATION_TARGET_ALPHA": RECALIBRATION_TARGET_ALPHA,
    "USE_SHARED_FOLDS": USE_SHARED_FOLDS,
    "FOLD_SEED": FOLD_SEED,
    "Y_MIN": Y_MIN,
    "Y_MAX": str(Y_MAX),  # inf is not native-YAML; store as string
}

config_path = os.path.join(EXPERIMENT_DIR, "global_config.txt")
with open(config_path, "w") as f:
    yaml.dump(global_config, f, default_flow_style=False, sort_keys=False)

print(f"Saved to: {config_path}")

In [ ]:
# ---- OOF metrics: AutoGluon (Model A leader) vs H2O AutoML leader ----
# Both vectors are out-of-fold predictions; metrics are UNWEIGHTED on both
# sides, and with USE_SHARED_FOLDS=True the folds are identical -> coherent.
h2o_oof_preds = (
    aml_h2o.leader.cross_validation_holdout_predictions()
    .as_data_frame()
    .iloc[:, 0]
    .to_numpy()
)

metrics_agl = evla.one_step_metrics(y_true=y_true, mu=mu_oof.values)
metrics_h2o = evla.one_step_metrics(y_true=y_true, mu=h2o_oof_preds)

metrics_df = pd.DataFrame(
    [
        {"model": "AutoGluon (Model A, leader)", **metrics_agl},
        {"model": "H2O AutoML (leader)", **metrics_h2o},
    ]
)[["model", "rmse", "mae", "mape", "r2"]]

print(metrics_df)

# ---- Leaderboards from both engines ----
leaderboard_agl = predictor_a.leaderboard(silent=True)
leaderboard_h2o = aml_h2o.leaderboard.as_data_frame()

# ---- Save everything for this experiment (TAG) into separate CSV files ----
metrics_df.to_csv(os.path.join(EXPERIMENT_DIR, "metrics_comparison.csv"), index=False)
leaderboard_agl.to_csv(os.path.join(EXPERIMENT_DIR, "leaderboard_autogluon.csv"), index=False)
leaderboard_h2o.to_csv(os.path.join(EXPERIMENT_DIR, "leaderboard_h2o.csv"), index=False)

print(f"Saved to: {EXPERIMENT_DIR}")

# 11. Load Artifacts & Predict (deployment simulation)

Loads ONLY what was persisted in Section 10 (`artifacts.pkl`) plus the two
predictors from disk — no in-session variable reused. This is what a deployment
script / the MBC controller would do.

In [ ]:
# ---- 1) Load the persisted artifact bundle (Section 10) ----
loaded_artifacts = load.load_pickle(EXPERIMENT_DIR, "artifacts.pkl")

# ---- 2) Reload both AutoGluon predictors from their TAG subfolders ----
predictor_a_loaded = TabularPredictor.load(
    os.path.join(EXPERIMENT_DIR, "model_a")
)
predictor_b_loaded = TabularPredictor.load(
    os.path.join(EXPERIMENT_DIR, "model_b")
)

print("Loaded artifacts keys:", list(loaded_artifacts.keys()))
print("Target:", loaded_artifacts["target"])
print("Features:", loaded_artifacts["features"])

In [ ]:
# ---- 3) Predict using ONLY the reloaded objects ----
X_query = df.head(5)

preds_loaded = modl.predict_with_uncertainty(
    X=X_query,
    predictor_a=predictor_a_loaded,
    predictor_b=predictor_b_loaded,
    weights=loaded_artifacts["weights"],
    model_names=loaded_artifacts["base_model_names"],
    features=loaded_artifacts["features"],
    variance_floor=loaded_artifacts["variance_floor"],
    recalibration_c=loaded_artifacts["recalibration_c"],
    use_weights=loaded_artifacts["use_weighted_variance"],
    y_min=loaded_artifacts["y_min"],
    y_max=loaded_artifacts["y_max"],
    ood_ref=loaded_artifacts["ood_ref"],
)
preds_loaded

In [ ]:
# ---- 4) Target-agnostic quick prediction from a plain dict ----
single_loaded = modl.predict_target(
    predictor_a_loaded,
    {"purity": 99.99, "initial_diameter": 23.0, "tensile_strength": 1200.0,
     "temperature": 473.0, "time": 60.0},
    features=loaded_artifacts["features"],
)
print(f"Predicted {loaded_artifacts['target']}: {single_loaded[0]:.1f} MPa")

In [ ]:
# ---- 5) Sanity check: reloaded predictions match the in-session ones ----
if "preds" in dir():
    max_mu_diff = float(np.abs(preds["mu"].to_numpy()
                               - preds_loaded["mu"].to_numpy()).max())
    max_sigma_diff = float(np.abs(preds["sigma_total"].to_numpy()
                                  - preds_loaded["sigma_total"].to_numpy()).max())
    print(f"max |mu diff|    = {max_mu_diff:.2e}")
    print(f"max |sigma diff| = {max_sigma_diff:.2e}")
    print("OK: artifacts reproduce the in-session pipeline."
          if max(max_mu_diff, max_sigma_diff) < 1e-8
          else "WARNING: mismatch - check Section 9 used the same ood_ref.")
else:
    print("Run Section 9 first to enable the sanity check.")